# Agent Trigger (Phase 3)

This notebook implements a **single Trigger agent producer** for person-state MQTT messages.

Phase 3 scope:
- Load simulation and MQTT settings from `config.yaml` using `simulated_city.config.load_config()`
- Simulate local movement with deterministic behavior
- Connect to MQTT via `mqtt.connect_mqtt(...)`
- Publish each person state via `mqtt.publish_json_checked(...)` to `${mqtt.base_topic}/pandemic/trigger/person_state`

In [ ]:
# Cell purpose: Import modules, load config, and resolve all simulation + MQTT constants.
from __future__ import annotations

import json

from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from random import Random
from time import sleep

import simulated_city.mqtt as mqtt
from simulated_city.config import load_config

cfg = load_config()
sim_cfg = cfg.simulation
mqtt_cfg = cfg.mqtt

if sim_cfg is None:
    raise ValueError("Phase 3 requires a 'simulation' section in config.yaml")

seed = sim_cfg.seed if sim_cfg.seed is not None else 42
rng = Random(seed)

time_step_s = sim_cfg.time_step_s
simulated_hours_per_step = sim_cfg.simulated_hours_per_step
publish_every_n_steps = sim_cfg.publish_every_n_steps
total_steps = sim_cfg.total_steps
step_delay_s = sim_cfg.step_delay_s

population_size = sim_cfg.population_size
initial_infected = min(sim_cfg.initial_infected, population_size)

min_lat = sim_cfg.bounds_min_lat
max_lat = sim_cfg.bounds_max_lat
min_lon = sim_cfg.bounds_min_lon
max_lon = sim_cfg.bounds_max_lon

max_speed_m_per_s = sim_cfg.max_speed_m_per_s

topic_person_state = f"{mqtt_cfg.base_topic}/pandemic/trigger/person_state"
topic_control_health_update = f"{mqtt_cfg.base_topic}/pandemic/control/health_update"

start_ts = sim_cfg.start_time if sim_cfg.start_time else datetime(2026, 1, 1, tzinfo=timezone.utc)

print(
    f"Loaded config. seed={seed}, population={population_size}, steps={total_steps}, "
    f"publish_every_n_steps={publish_every_n_steps}"
)
print(f"MQTT target: {mqtt_cfg.host}:{mqtt_cfg.port} topic={topic_person_state}")
print(f"Trigger feedback subscription: {topic_control_health_update}")

In [ ]:
# Cell purpose: Build initial in-memory person states using config-driven geometry and population values.
@dataclass
class PersonState:
    person_id: str
    lat: float
    lon: float
    dlat: float
    dlon: float
    health_status: str


delta_scale = max(0.00001, (max_speed_m_per_s * time_step_s) / 111_000.0)

persons: list[PersonState] = []
for index in range(population_size):
    lat = rng.uniform(min_lat, max_lat)
    lon = rng.uniform(min_lon, max_lon)
    dlat = rng.uniform(-delta_scale, delta_scale)
    dlon = rng.uniform(-delta_scale, delta_scale)
    health_status = "infected" if index < initial_infected else "susceptible"
    persons.append(
        PersonState(
            person_id=f"person-{index:03d}",
            lat=lat,
            lon=lon,
            dlat=dlat,
            dlon=dlon,
            health_status=health_status,
        )
    )

infected_count = sum(1 for p in persons if p.health_status == "infected")
susceptible_count = sum(1 for p in persons if p.health_status == "susceptible")
print(
    f"Initialized {len(persons)} persons. infected={infected_count}, susceptible={susceptible_count}, "
    f"delta_scale={delta_scale:.8f}"
)

In [ ]:
# Cell purpose: Connect to MQTT and publish each person_state message to the Trigger topic.
def _reflect(value: float, velocity: float, low: float, high: float) -> tuple[float, float]:
    next_value = value + velocity
    next_velocity = velocity

    if next_value < low:
        overflow = low - next_value
        next_value = low + overflow
        next_velocity = -next_velocity
    elif next_value > high:
        overflow = next_value - high
        next_value = high - overflow
        next_velocity = -next_velocity

    return next_value, next_velocity


client = mqtt.connect_mqtt(mqtt_cfg, client_id_suffix="trigger")
print(f"Connected to MQTT broker at {mqtt_cfg.host}:{mqtt_cfg.port}")

valid_health_statuses = {"susceptible", "infected", "recovered"}
pending_health_updates: dict[str, str] = {}
feedback_messages_received = 0
applied_feedback_updates = 0

def on_health_update(client_obj, userdata, msg):
    global feedback_messages_received
    if msg.topic != topic_control_health_update:
        return

    try:
        payload = json.loads(msg.payload.decode())
    except Exception:
        return

    person_id = payload.get("person_id")
    to_status = payload.get("to_status")
    if not isinstance(person_id, str):
        return
    if to_status not in valid_health_statuses:
        return

    pending_health_updates[person_id] = to_status
    feedback_messages_received += 1


client.on_message = on_health_update
client.subscribe(topic_control_health_update)
print(f"Subscribed to {topic_control_health_update} for feedback updates")

published_messages = 0
print(f"Simulation start UTC: {start_ts.isoformat()}")

for step in range(total_steps):
    moved: list[PersonState] = []
    step_ts = (start_ts + timedelta(hours=simulated_hours_per_step * step)).isoformat()

    for person in persons:
        feedback_status = pending_health_updates.pop(person.person_id, None)
        current_status = feedback_status if feedback_status is not None else person.health_status
        if feedback_status is not None and feedback_status != person.health_status:
            applied_feedback_updates += 1

        next_lat, next_dlat = _reflect(person.lat, person.dlat, min_lat, max_lat)
        next_lon, next_dlon = _reflect(person.lon, person.dlon, min_lon, max_lon)
        next_person = PersonState(
            person_id=person.person_id,
            lat=next_lat,
            lon=next_lon,
            dlat=next_dlat,
            dlon=next_dlon,
            health_status=current_status,
        )
        moved.append(next_person)

        if step % publish_every_n_steps == 0:
            payload = {
                "step": step,
                "ts": step_ts,
                "person_id": next_person.person_id,
                "lat": round(next_person.lat, 6),
                "lon": round(next_person.lon, 6),
                "health_status": next_person.health_status,
            }
            mqtt.publish_json_checked(client, topic_person_state, payload)
            published_messages += 1

    persons = moved

    infected_count = sum(1 for p in persons if p.health_status == "infected")
    recovered_count = sum(1 for p in persons if p.health_status == "recovered")
    susceptible_count = len(persons) - infected_count - recovered_count

    should_print_step = (step < 3) or (step == total_steps - 1)
    if should_print_step:
        sample = persons[0]
        print(
            f"step={step:04d} sample={sample.person_id} lat={sample.lat:.6f} lon={sample.lon:.6f} "
            f"susceptible={susceptible_count} infected={infected_count} recovered={recovered_count} "
            f"feedback_applied={applied_feedback_updates}"
        )

    if step_delay_s > 0:
        sleep(step_delay_s)

print(
    f"Phase 3 publish complete. published_messages={published_messages} topic={topic_person_state} "
    f"feedback_messages_received={feedback_messages_received} feedback_applied={applied_feedback_updates}"
)

In [ ]:
# Cell purpose: Show final payload preview and disconnect cleanly from MQTT.
sample_person = persons[0]
preview_message = {
    "step": total_steps - 1,
    "ts": (start_ts + timedelta(hours=simulated_hours_per_step * (total_steps - 1))).isoformat(),
    "person_id": sample_person.person_id,
    "lat": round(sample_person.lat, 6),
    "lon": round(sample_person.lon, 6),
    "health_status": sample_person.health_status,
}
print(preview_message)

connector = getattr(client, "_simulated_city_connector", None)
if connector is not None:
    connector.disconnect()
    print("Disconnected from MQTT broker.")
else:
    print("No connector reference found; client disconnect skipped.")